⚠️ **Gemini Parse Error** — response could not be parsed as a valid notebook.
Raw output preserved below for manual recovery.

In [ ]:
{
  "nbformat": 4,
  "nbformat_minor": 0,
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "python",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "nbconvert_exporter": "python",
      "pygments_lexer": "ipython3",
      "version": "3.9.18"
    }
  },
  "cells": [
    {
      "cell_type": "markdown",
      "source": [
        "# ODI to Databricks Migration: TRG_LOC Insert\n",
        "\n",
        "**Conversion Timestamp:** 2024-07-30 12:00:00\n",
        "\n",
        "This notebook migrates a simple `INSERT` statement for the `TRG_LOC` table from Oracle ODI to Databricks Spark SQL. It includes standard ETL parameter widgets as per migration guidelines."
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {
        "collapsed": false
      },
      "outputs": [],
      "source": [
        "dbutils.widgets.text(\"ETL_JOB_TYPE\", \"\", \"1. ETL Job Type\")\n",
        "dbutils.widgets.text(\"DATASOURCE_NUM_ID\", \"\", \"2. Datasource Number ID\")\n",
        "dbutils.widgets.text(\"ETL_PROC_WID\", \"\", \"3. ETL Process WID\")\n",
        "dbutils.widgets.text(\"ODI_SESS_NO\", \"\", \"4. ODI Session Number\")"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# ETL Parameters"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS\n",
        "SELECT\n",
        "  '${ETL_JOB_TYPE}' AS etl_job_type,\n",
        "  '${DATASOURCE_NUM_ID}' AS datasource_num_id,\n",
        "  '${ETL_PROC_WID}' AS etl_proc_wid,\n",
        "  '${ODI_SESS_NO}' AS odi_sess_no;"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "display(spark.sql(\"SELECT * FROM v_etl_parameters\"))"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Main INSERT Statement"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "-- SCEN_TASK_NO in {10}, {20}, {30}: These are ODI execution markers without associated SQL to convert.\n",
        "INSERT INTO workspace.hr.trg_loc\n",
        "(\n",
        "  location_id ,\n",
        "  street_address ,\n",
        "  postal_code ,\n",
        "  city ,\n",
        "  state_province ,\n",
        "  country_id \n",
        ") \n",
        "SELECT \n",
        "  locations.location_id ,\n",
        "  locations.street_address ,\n",
        "  locations.postal_code ,\n",
        "  locations.city ,\
        "  locations.state_province ,\
        "  locations.country_id  \n",
        "FROM \n",
        "  workspace.hr.locations AS locations;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Validation"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "metadata": {},
      "outputs": [],
      "source": [
        "-- MAGIC %sql\n",
        "SELECT COUNT(*) AS total_records_in_trg_loc FROM workspace.hr.trg_loc;"
      ]
    },
    {
      "cell_type": "markdown",
      "source": [
        "# Conversion Notes and Manual Actions Required\n",
        "\n",
        "1.  **Schema and Table Naming:** Oracle `HR.TRG_LOC` and `HR.LOCATIONS` have been converted to `workspace.hr.trg_loc` and `workspace.hr.locations` respectively, following the `workspace.{schema_lowercase}.{table_lowercase}` convention.\n",
        "2.  **Oracle Hints Removed:** The `/*+ APPEND PARALLEL */` Oracle hint has been removed as it is not applicable in Databricks Spark SQL for Delta tables.\n",
        "3.  **Column Lowercasing:** All column names in the `INSERT` statement are lowercased for consistency with Spark SQL best practices.\n",
        "4.  **ODI Markers:** `SCEN_TASK_NO` entries without SQL have been noted but not converted into separate cells.\n",
        "5.  **Target Table DDL:** Ensure that `workspace.hr.trg_loc` has its DDL defined and is created prior to this notebook's execution, matching the expected data types from `workspace.hr.locations`.\n",
        "6.  **Parameter Widgets:** Standard ETL parameter widgets have been included, although not directly used by this specific `INSERT` statement. This is to maintain consistency with general migration patterns."
      ]
    }
  ]
}